In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("DeltaLakeTransactionsLab") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.sql.warehouse.dir", "/home/jovyan/spark-warehouse") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0") \
    .getOrCreate()

print("Done")

Done


In [1]:
from IPython.core.magic import register_cell_magic

@register_cell_magic
def sql(line, cell):
    statements = [s.strip() for s in cell.split(";") if s.strip()]
    
    result = None
    for s in statements:
        result = spark.sql(s)
        if result is not None:
            try:
                result.show(truncate=False)
            except Exception:
                pass

In [4]:
%%sql
SET spark.databricks.delta.properties.defaults.autoOptimize.optimizeWrite = false;
SET spark.databricks.delta.properties.defaults.autoOptimize.autoCompact = false;

+---------------------------------------------------------------------+-----+
|key                                                                  |value|
+---------------------------------------------------------------------+-----+
|spark.databricks.delta.properties.defaults.autoOptimize.optimizeWrite|false|
+---------------------------------------------------------------------+-----+

+-------------------------------------------------------------------+-----+
|key                                                                |value|
+-------------------------------------------------------------------+-----+
|spark.databricks.delta.properties.defaults.autoOptimize.autoCompact|false|
+-------------------------------------------------------------------+-----+



In [5]:
%%sql
CREATE OR REPLACE TABLE pracownicy (
    id INT,
    nazwisko STRING,
    pensja LONG
) USING delta
TBLPROPERTIES ('delta.enableDeletionVectors' = true);

ALTER TABLE pracownicy ADD CONSTRAINT Ids CHECK (id > 0 and id < 999999);

INSERT OVERWRITE pracownicy VALUES
    ('1', 'Kowalski', 5000),
    ('2', 'Nowak', 6000),
    ('3', 'Zieliński', 7000);

SELECT * FROM pracownicy;

++
||
++
++

++
||
++
++

++
||
++
++

+---+---------+------+
|id |nazwisko |pensja|
+---+---------+------+
|3  |Zieliński|7000  |
|1  |Kowalski |5000  |
|2  |Nowak    |6000  |
+---+---------+------+



In [6]:
%%sql
DESCRIBE EXTENDED pracownicy;

+----------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------+
|col_name                    |data_type                                                                                                                                                                                                                                                                                     |comment|
+----------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------+
|id                   

In [7]:
!ls -R /home/jovyan/spark-warehouse/pracownicy

/home/jovyan/spark-warehouse/pracownicy:
_delta_log
part-00000-31fe689e-a0ae-4c75-809d-d653b84598eb-c000.snappy.parquet
part-00001-0557b4a2-bb31-483d-8d45-012f29701e25-c000.snappy.parquet
part-00002-f90f7efd-6a6d-40d0-ab6c-c0a935c53add-c000.snappy.parquet

/home/jovyan/spark-warehouse/pracownicy/_delta_log:
00000000000000000000.json  00000000000000000002.json
00000000000000000001.json  _commits

/home/jovyan/spark-warehouse/pracownicy/_delta_log/_commits:


In [9]:
%%sql
SELECT * FROM JSON.`/home/jovyan/spark-warehouse/pracownicy/_delta_log/00000000000000000001.json`;

+-----------------------------------------------------------------------------------------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------------------------------------------+
|commitInfo                                                                                                                                                       |metaData                                                                                                                                                                                                                                                                            

In [10]:
%%sql
INSERT INTO pracownicy VALUES
    ('4', 'Kowalski', 5000),
    ('-1', 'Zieliński', 7000);

Py4JJavaError: An error occurred while calling o35.sql.
: org.apache.spark.sql.delta.schema.DeltaInvariantViolationException: [DELTA_VIOLATE_CONSTRAINT_WITH_VALUES] CHECK constraint ids ((id > 0) AND (id < 999999)) violated by row with values:
 - id : -1
	at org.apache.spark.sql.delta.schema.DeltaInvariantViolationException$.getConstraintViolationWithValuesException(InvariantViolationException.scala:75)
	at org.apache.spark.sql.delta.schema.DeltaInvariantViolationException$.apply(InvariantViolationException.scala:101)
	at org.apache.spark.sql.delta.schema.DeltaInvariantViolationException$.apply(InvariantViolationException.scala:112)
	at org.apache.spark.sql.delta.schema.DeltaInvariantViolationException.apply(InvariantViolationException.scala)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$SpecificUnsafeProjection.CheckDeltaInvariant_0$(Unknown Source)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$SpecificUnsafeProjection.apply(Unknown Source)
	at org.apache

In [11]:
!ls -R /home/jovyan/spark-warehouse/pracownicy

/home/jovyan/spark-warehouse/pracownicy:
_delta_log
part-00000-31fe689e-a0ae-4c75-809d-d653b84598eb-c000.snappy.parquet
part-00000-a9ef72e3-4afd-4d73-9362-1c473b03a2a8-c000.snappy.parquet
part-00001-0557b4a2-bb31-483d-8d45-012f29701e25-c000.snappy.parquet
part-00001-d6a704d4-c9c0-4b0a-a03b-e215647196c1-c000.snappy.parquet
part-00002-f90f7efd-6a6d-40d0-ab6c-c0a935c53add-c000.snappy.parquet

/home/jovyan/spark-warehouse/pracownicy/_delta_log:
00000000000000000000.json  00000000000000000002.json
00000000000000000001.json  _commits

/home/jovyan/spark-warehouse/pracownicy/_delta_log/_commits:


In [12]:
%%sql
SELECT * FROM pracownicy;

+---+---------+------+
|id |nazwisko |pensja|
+---+---------+------+
|3  |Zieliński|7000  |
|1  |Kowalski |5000  |
|2  |Nowak    |6000  |
+---+---------+------+



In [13]:
%%sql
DESC HISTORY pracownicy;

+-------+-----------------------+------+--------+-----------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+--------+---------+-----------+--------------+-------------+-----------------------------------------------------------+------------+-----------------------------------+
|version|timestamp              |userId|userName|operation              |operationParameters                                                                                                                                                                                                   |job |notebook|clusterId|readVersion|isolationLevel|isBlindAppend|operationMetrics                                           |userMetadata|engineInfo                         |
+-------+-----------------------+------+--------+-----------------------+-

In [14]:
%%sql
UPDATE pracownicy
SET pensja = 8000
WHERE nazwisko = 'Zieliński';

SELECT * FROM pracownicy;

+-----------------+
|num_affected_rows|
+-----------------+
|1                |
+-----------------+

+---+---------+------+
|id |nazwisko |pensja|
+---+---------+------+
|3  |Zieliński|8000  |
|1  |Kowalski |5000  |
|2  |Nowak    |6000  |
+---+---------+------+



In [15]:
!ls -R /home/jovyan/spark-warehouse/pracownicy

/home/jovyan/spark-warehouse/pracownicy:
deletion_vector_41201112-24ef-4e2f-b2d8-78c779bbe117.bin
_delta_log
part-00000-31fe689e-a0ae-4c75-809d-d653b84598eb-c000.snappy.parquet
part-00000-a9ef72e3-4afd-4d73-9362-1c473b03a2a8-c000.snappy.parquet
part-00000-ae90771e-dd69-4154-9a10-3798195250a0-c000.snappy.parquet
part-00001-0557b4a2-bb31-483d-8d45-012f29701e25-c000.snappy.parquet
part-00001-d6a704d4-c9c0-4b0a-a03b-e215647196c1-c000.snappy.parquet
part-00002-f90f7efd-6a6d-40d0-ab6c-c0a935c53add-c000.snappy.parquet

/home/jovyan/spark-warehouse/pracownicy/_delta_log:
00000000000000000000.json  00000000000000000002.json  _commits
00000000000000000001.json  00000000000000000003.json

/home/jovyan/spark-warehouse/pracownicy/_delta_log/_commits:


In [16]:
%%sql
SELECT * FROM JSON.`/home/jovyan/spark-warehouse/pracownicy/_delta_log/00000000000000000003.json`;

+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------+
|add                                                                                                                                                                                                                                                                                                  |commitInfo                                                                      

In [17]:
import time
from pyspark.sql.functions import udf
from pyspark.sql.types import IntegerType

def f(pensja):
    time.sleep(4)
    return pensja

spark.udf.register("fake_delay", f, IntegerType())


<function __main__.f(pensja)>

In [18]:
import threading

def transakcja_A():
    print("[Wątek A] Długa modyfikacja...")
    try:
        # 1. tymczasowy widok w Sparku zawierający opóźnioną wartość.
        spark.sql("SELECT '1' as id, fake_delay(7000) as nowa_pensja").createOrReplaceTempView("v_wyplata_A")
        
        # 2. MERGE korzystając z tego widoku. 
        spark.sql("""
            MERGE INTO pracownicy t
            USING v_wyplata_A s
            ON t.id = s.id
            WHEN MATCHED THEN UPDATE SET t.pensja = s.nowa_pensja
        """)
        print("\nTransakcja A zapisana.")
    except Exception as e:
        print("\nTransakcja A odrzucona")
        e = str(e)
        r = [line for line in e.splitlines() if "Concurrent" in line or "Exception" in line]
        if r:
            print(r[0].strip())
        else:
            print("\n".join(r.splitlines()[:3]))

def transakcja_B():
    time.sleep(1)
    print("[Wątek B] Szybka modyfikacja w tym samym czasie")
    try:
        spark.sql("UPDATE pracownicy SET pensja = 8888 WHERE id = '1'")
        print("Transakcja B zapisana.")
    except Exception as e:
        print(f"\nBłąd transakcji B")

watek_a = threading.Thread(target=transakcja_A)
watek_b = threading.Thread(target=transakcja_B)

watek_a.start()
watek_b.start()

watek_a.join()
watek_b.join()

[Wątek A] Długa modyfikacja...
[Wątek B] Szybka modyfikacja w tym samym czasie
Transakcja B zapisana.

Transakcja A odrzucona
: io.delta.exceptions.ConcurrentAppendException: [DELTA_CONCURRENT_APPEND] ConcurrentAppendException: Files were added to the root of the table by a concurrent update. Please try the operation again.


In [19]:
!ls -R /home/jovyan/spark-warehouse/pracownicy

/home/jovyan/spark-warehouse/pracownicy:
deletion_vector_41201112-24ef-4e2f-b2d8-78c779bbe117.bin
deletion_vector_a54ecde4-1195-47ec-b9e9-a3c762cebdb4.bin
deletion_vector_a64e7ba3-a4b8-49fd-b27d-b4cd412bf6d1.bin
_delta_log
part-00000-31fe689e-a0ae-4c75-809d-d653b84598eb-c000.snappy.parquet
part-00000-38375481-85b9-4bd3-9786-d621efa3684b-c000.snappy.parquet
part-00000-48fba62b-58da-47e2-8b34-1a1ff407841b-c000.snappy.parquet
part-00000-a9ef72e3-4afd-4d73-9362-1c473b03a2a8-c000.snappy.parquet
part-00000-ae90771e-dd69-4154-9a10-3798195250a0-c000.snappy.parquet
part-00001-0557b4a2-bb31-483d-8d45-012f29701e25-c000.snappy.parquet
part-00001-d6a704d4-c9c0-4b0a-a03b-e215647196c1-c000.snappy.parquet
part-00002-f90f7efd-6a6d-40d0-ab6c-c0a935c53add-c000.snappy.parquet

/home/jovyan/spark-warehouse/pracownicy/_delta_log:
00000000000000000000.json  00000000000000000002.json  00000000000000000004.json
00000000000000000001.json  00000000000000000003.json  _commits

/home/jovyan/spark-warehouse/pracown

In [20]:
from pyspark.sql.types import StructType, StructField, StringType, LongType, IntegerType

pracownik = StructType([
    StructField("id", IntegerType(), True),
    StructField("nazwisko", StringType(), True),
    StructField("pensja", LongType(), True),
    StructField("stanowisko", StringType(), True)
])

df_csv = spark.read \
    .option("header", "true") \
    .option("sep", ",") \
    .schema(pracownik) \
    .csv("pracownicy.csv")  

df_csv.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("pracownicy_updates")



In [21]:
%%sql
SELECT * FROM pracownicy_updates;

+---+----------+------+--------------------+
|id |nazwisko  |pensja|stanowisko          |
+---+----------+------+--------------------+
|3  |Zieliński |7000  |HR Specialist       |
|4  |Kaczmarek |10308 |Project Manager     |
|5  |Lorenc    |19234 |DevOps Engineer     |
|6  |Frankowski|13952 |Project Manager     |
|7  |Maćkowiak |11611 |Software Engineer   |
|8  |Tokarz    |8895  |Junior Data Engineer|
|9  |Pałka     |8991  |Data Analyst        |
|10 |Stasiak   |7121  |HR Specialist       |
|11 |Kosowski  |15941 |Software Engineer   |
|12 |Kępa      |10310 |Data Engineer       |
|13 |Kogut     |12830 |DevOps Engineer     |
|14 |Kaczor    |7294  |Junior Data Engineer|
|15 |Kubat     |14732 |Software Engineer   |
|16 |Lipka     |6319  |HR Specialist       |
|17 |Wróblewski|24021 |Senior Data Engineer|
|18 |Kałuża    |7971  |HR Specialist       |
|19 |Kępa      |11639 |Software Engineer   |
|20 |Furman    |15730 |Data Engineer       |
|21 |Czajka    |9929  |Software Engineer   |
|22 |Frank

In [25]:
from delta.tables import DeltaTable

target_table = DeltaTable.forName(spark, "pracownicy")

source_df = spark.table("pracownicy_updates")
(
target_table.alias("t")
    .merge(
        source_df.alias("s"),
        condition="t.id = s.id"
    )
    .withSchemaEvolution()
    #.whenMatchedUpdateAll()
    #.whenNotMatchedInsertAll()
    .whenMatchedUpdate(
        set={
            "nazwisko": "s.nazwisko",
            "pensja": "s.pensja",
            "stanowisko": "s.stanowisko"
        }
    )
    .whenNotMatchedInsert(
        values={
            "id": "s.id",
            "nazwisko": "s.nazwisko",
            "pensja": "s.pensja",
            "stanowisko": "s.stanowisko"
        }
    )
    .execute()
)

In [23]:
%%sql
MERGE INTO pracownicy
USING pracownicy_updates
ON pracownicy.id = pracownicy_updates.id
WHEN MATCHED THEN
    UPDATE SET *
WHEN NOT MATCHED THEN
    INSERT *

+-----------------+----------------+----------------+-----------------+
|num_affected_rows|num_updated_rows|num_deleted_rows|num_inserted_rows|
+-----------------+----------------+----------------+-----------------+
|50001            |50001           |0               |0                |
+-----------------+----------------+----------------+-----------------+



In [26]:
%%sql
SELECT * FROM pracownicy
WHERE id < 10;


+---+----------+------+--------------------+
|id |nazwisko  |pensja|stanowisko          |
+---+----------+------+--------------------+
|3  |Zieliński |7000  |HR Specialist       |
|4  |Kaczmarek |10308 |Project Manager     |
|5  |Lorenc    |19234 |DevOps Engineer     |
|6  |Frankowski|13952 |Project Manager     |
|7  |Maćkowiak |11611 |Software Engineer   |
|8  |Tokarz    |8895  |Junior Data Engineer|
|9  |Pałka     |8991  |Data Analyst        |
|1  |Kowalski  |8888  |NULL                |
|2  |Nowak     |6000  |NULL                |
+---+----------+------+--------------------+



In [29]:
!ls -R /home/jovyan/spark-warehouse/pracownicy

/home/jovyan/spark-warehouse/pracownicy:
deletion_vector_41201112-24ef-4e2f-b2d8-78c779bbe117.bin
deletion_vector_4c323ccc-cae1-4480-b5bd-9963aa976f47.bin
deletion_vector_8f5f1e71-1dfb-4dbf-b406-5cc69fa66729.bin
deletion_vector_8f99b508-e65b-44d4-839a-a6963a6defe7.bin
deletion_vector_920e9228-90f0-4b8b-becf-c436163b133c.bin
deletion_vector_a54ecde4-1195-47ec-b9e9-a3c762cebdb4.bin
deletion_vector_a64e7ba3-a4b8-49fd-b27d-b4cd412bf6d1.bin
_delta_log
part-00000-0dc12878-954d-4654-9f8d-3d2a80e2856a-c000.snappy.parquet
part-00000-31fe689e-a0ae-4c75-809d-d653b84598eb-c000.snappy.parquet
part-00000-32dd5f20-c7e8-4d57-850b-a55f6332ca1c-c000.snappy.parquet
part-00000-38375481-85b9-4bd3-9786-d621efa3684b-c000.snappy.parquet
part-00000-48fba62b-58da-47e2-8b34-1a1ff407841b-c000.snappy.parquet
part-00000-9aac8332-5c53-4610-a437-233c4eee4c3d-c000.snappy.parquet
part-00000-a9ef72e3-4afd-4d73-9362-1c473b03a2a8-c000.snappy.parquet
part-00000-ad856ff4-2850-4757-81a0-78e4747c1b35-c000.snappy.parquet
part-

In [30]:
%%sql
ALTER TABLE pracownicy 
SET TBLPROPERTIES (delta.enableChangeDataFeed = true)

++
||
++
++



In [31]:
%%sql
SELECT * FROM pracownicy
WHERE pracownicy.pensja > 22000;

DELETE FROM pracownicy
WHERE pracownicy.pensja > 22000;

+---+----------+------+--------------------+
|id |nazwisko  |pensja|stanowisko          |
+---+----------+------+--------------------+
|17 |Wróblewski|24021 |Senior Data Engineer|
|37 |Godlewski |23427 |Senior Data Engineer|
|86 |Kmiecik   |22146 |Senior Data Engineer|
|114|Michalski |24619 |Senior Data Engineer|
|126|Karaś     |22237 |Senior Data Engineer|
|136|Wróblewski|23632 |Senior Data Engineer|
|146|Żuk       |25344 |Senior Data Engineer|
|154|Łuczak    |24075 |Senior Data Engineer|
|163|Łuczak    |24170 |Senior Data Engineer|
|191|Maj       |22178 |Senior Data Engineer|
|198|Jaworski  |23844 |Senior Data Engineer|
|212|Jasiński  |22244 |Senior Data Engineer|
|219|Nowak     |22854 |Senior Data Engineer|
|220|Krakowiak |22375 |Senior Data Engineer|
|292|Nowakowski|22603 |Senior Data Engineer|
|306|Prus      |23571 |Senior Data Engineer|
|314|Kozieł    |23125 |Senior Data Engineer|
|345|Wieczorek |24590 |Senior Data Engineer|
|350|Janicki   |24310 |Senior Data Engineer|
|373|Zawad

In [32]:
!ls -R /home/jovyan/spark-warehouse/pracownicy

/home/jovyan/spark-warehouse/pracownicy:
deletion_vector_41201112-24ef-4e2f-b2d8-78c779bbe117.bin
deletion_vector_4c323ccc-cae1-4480-b5bd-9963aa976f47.bin
deletion_vector_83cb6812-29bb-4ac0-9044-925bd1692139.bin
deletion_vector_8f5f1e71-1dfb-4dbf-b406-5cc69fa66729.bin
deletion_vector_8f99b508-e65b-44d4-839a-a6963a6defe7.bin
deletion_vector_920e9228-90f0-4b8b-becf-c436163b133c.bin
deletion_vector_a54ecde4-1195-47ec-b9e9-a3c762cebdb4.bin
deletion_vector_a64e7ba3-a4b8-49fd-b27d-b4cd412bf6d1.bin
_delta_log
part-00000-0dc12878-954d-4654-9f8d-3d2a80e2856a-c000.snappy.parquet
part-00000-31fe689e-a0ae-4c75-809d-d653b84598eb-c000.snappy.parquet
part-00000-32dd5f20-c7e8-4d57-850b-a55f6332ca1c-c000.snappy.parquet
part-00000-38375481-85b9-4bd3-9786-d621efa3684b-c000.snappy.parquet
part-00000-48fba62b-58da-47e2-8b34-1a1ff407841b-c000.snappy.parquet
part-00000-9aac8332-5c53-4610-a437-233c4eee4c3d-c000.snappy.parquet
part-00000-a9ef72e3-4afd-4d73-9362-1c473b03a2a8-c000.snappy.parquet
part-00000-ad856

In [33]:
%%sql
DESCRIBE HISTORY pracownicy;

+-------+-----------------------+------+--------+-----------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+--------+---------+-----------+--------------+-------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [35]:
%%sql
SELECT * FROM table_changes('pracownicy', 9, 10);

+---+----------+------+--------------------+------------+---------------+-----------------------+
|id |nazwisko  |pensja|stanowisko          |_change_type|_commit_version|_commit_timestamp      |
+---+----------+------+--------------------+------------+---------------+-----------------------+
|17 |Wróblewski|24021 |Senior Data Engineer|delete      |10             |2026-06-02 19:29:18.641|
|37 |Godlewski |23427 |Senior Data Engineer|delete      |10             |2026-06-02 19:29:18.641|
|86 |Kmiecik   |22146 |Senior Data Engineer|delete      |10             |2026-06-02 19:29:18.641|
|114|Michalski |24619 |Senior Data Engineer|delete      |10             |2026-06-02 19:29:18.641|
|126|Karaś     |22237 |Senior Data Engineer|delete      |10             |2026-06-02 19:29:18.641|
|136|Wróblewski|23632 |Senior Data Engineer|delete      |10             |2026-06-02 19:29:18.641|
|146|Żuk       |25344 |Senior Data Engineer|delete      |10             |2026-06-02 19:29:18.641|
|154|Łuczak    |2407

In [36]:
%%sql
SELECT * FROM table_changes('pracownicy', 10)
WHERE _change_type = 'delete';

+---+----------+------+--------------------+------------+---------------+-----------------------+
|id |nazwisko  |pensja|stanowisko          |_change_type|_commit_version|_commit_timestamp      |
+---+----------+------+--------------------+------------+---------------+-----------------------+
|17 |Wróblewski|24021 |Senior Data Engineer|delete      |10             |2026-06-02 19:29:18.641|
|37 |Godlewski |23427 |Senior Data Engineer|delete      |10             |2026-06-02 19:29:18.641|
|86 |Kmiecik   |22146 |Senior Data Engineer|delete      |10             |2026-06-02 19:29:18.641|
|114|Michalski |24619 |Senior Data Engineer|delete      |10             |2026-06-02 19:29:18.641|
|126|Karaś     |22237 |Senior Data Engineer|delete      |10             |2026-06-02 19:29:18.641|
|136|Wróblewski|23632 |Senior Data Engineer|delete      |10             |2026-06-02 19:29:18.641|
|146|Żuk       |25344 |Senior Data Engineer|delete      |10             |2026-06-02 19:29:18.641|
|154|Łuczak    |2407

In [41]:
%%sql
SELECT * FROM PRACOWNICY 
where pracownicy.pensja > 22000;

SELECT * FROM pracownicy VERSION AS OF 9
WHERE pracownicy.pensja > 22000;

SELECT * FROM pracownicy TIMESTAMP AS OF '2026-06-02 19:28:50'
WHERE pracownicy.pensja > 22000;

+---+--------+------+----------+
|id |nazwisko|pensja|stanowisko|
+---+--------+------+----------+
+---+--------+------+----------+

+---+----------+------+--------------------+
|id |nazwisko  |pensja|stanowisko          |
+---+----------+------+--------------------+
|17 |Wróblewski|24021 |Senior Data Engineer|
|37 |Godlewski |23427 |Senior Data Engineer|
|86 |Kmiecik   |22146 |Senior Data Engineer|
|114|Michalski |24619 |Senior Data Engineer|
|126|Karaś     |22237 |Senior Data Engineer|
|136|Wróblewski|23632 |Senior Data Engineer|
|146|Żuk       |25344 |Senior Data Engineer|
|154|Łuczak    |24075 |Senior Data Engineer|
|163|Łuczak    |24170 |Senior Data Engineer|
|191|Maj       |22178 |Senior Data Engineer|
|198|Jaworski  |23844 |Senior Data Engineer|
|212|Jasiński  |22244 |Senior Data Engineer|
|219|Nowak     |22854 |Senior Data Engineer|
|220|Krakowiak |22375 |Senior Data Engineer|
|292|Nowakowski|22603 |Senior Data Engineer|
|306|Prus      |23571 |Senior Data Engineer|
|314|Kozieł 

In [42]:
%%sql
RESTORE TABLE pracownicy TO VERSION AS OF 9;

SELECT * FROM PRACOWNICY 
WHERE pracownicy.pensja > 22000;

+------------------------+--------------------------+-----------------+------------------+------------------+-------------------+
|table_size_after_restore|num_of_files_after_restore|num_removed_files|num_restored_files|removed_files_size|restored_files_size|
+------------------------+--------------------------+-----------------+------------------+------------------+-------------------+
|431623                  |3                         |1                |1                 |429680            |429680             |
+------------------------+--------------------------+-----------------+------------------+------------------+-------------------+

+---+----------+------+--------------------+
|id |nazwisko  |pensja|stanowisko          |
+---+----------+------+--------------------+
|17 |Wróblewski|24021 |Senior Data Engineer|
|37 |Godlewski |23427 |Senior Data Engineer|
|86 |Kmiecik   |22146 |Senior Data Engineer|
|114|Michalski |24619 |Senior Data Engineer|
|126|Karaś     |22237 |Senior Data

In [43]:
!ls -R /home/jovyan/spark-warehouse/pracownicy

/home/jovyan/spark-warehouse/pracownicy:
deletion_vector_41201112-24ef-4e2f-b2d8-78c779bbe117.bin
deletion_vector_4c323ccc-cae1-4480-b5bd-9963aa976f47.bin
deletion_vector_83cb6812-29bb-4ac0-9044-925bd1692139.bin
deletion_vector_8f5f1e71-1dfb-4dbf-b406-5cc69fa66729.bin
deletion_vector_8f99b508-e65b-44d4-839a-a6963a6defe7.bin
deletion_vector_920e9228-90f0-4b8b-becf-c436163b133c.bin
deletion_vector_a54ecde4-1195-47ec-b9e9-a3c762cebdb4.bin
deletion_vector_a64e7ba3-a4b8-49fd-b27d-b4cd412bf6d1.bin
_delta_log
part-00000-0dc12878-954d-4654-9f8d-3d2a80e2856a-c000.snappy.parquet
part-00000-31fe689e-a0ae-4c75-809d-d653b84598eb-c000.snappy.parquet
part-00000-32dd5f20-c7e8-4d57-850b-a55f6332ca1c-c000.snappy.parquet
part-00000-38375481-85b9-4bd3-9786-d621efa3684b-c000.snappy.parquet
part-00000-48fba62b-58da-47e2-8b34-1a1ff407841b-c000.snappy.parquet
part-00000-9aac8332-5c53-4610-a437-233c4eee4c3d-c000.snappy.parquet
part-00000-a9ef72e3-4afd-4d73-9362-1c473b03a2a8-c000.snappy.parquet
part-00000-ad856

In [55]:

for i in range(25):
    spark.sql(f"INSERT INTO pracownicy VALUES \
                ({i + 50002}, \
                'Kowalski', \
                '5000', \
                'HR Specialist' \
                )")


In [64]:
import os
folder_path = "/home/jovyan/spark-warehouse/pracownicy"

if os.path.exists(folder_path):
    files = [f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))]
    print(len(files))

2


In [60]:
%%sql
OPTIMIZE pracownicy ZORDER BY (stanowisko);

+--------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|path                                        |metrics                                                                                                                                                                                                                        |
+--------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|file:/home/jovyan/spark-warehouse/pracownicy|{1, 26, {430402, 430402, 430402.0, 1, 430402}, {1296, 430302, 17796.23076923077, 26, 462702}, 1, {all, {0, 0}, {26, 462702}, 0, {26, 462702},

In [62]:
%%sql
SET spark.sql.files.ignoreMissingFiles = false;
SET spark.databricks.delta.retentionDurationCheck.enabled = false;

VACUUM pracownicy RETAIN 0 HOURS;

+----------------------------------+-----+
|key                               |value|
+----------------------------------+-----+
|spark.sql.files.ignoreMissingFiles|false|
+----------------------------------+-----+

+-----------------------------------------------------+-----+
|key                                                  |value|
+-----------------------------------------------------+-----+
|spark.databricks.delta.retentionDurationCheck.enabled|false|
+-----------------------------------------------------+-----+

+--------------------------------------------+
|path                                        |
+--------------------------------------------+
|file:/home/jovyan/spark-warehouse/pracownicy|
+--------------------------------------------+



In [63]:
!ls -R /home/jovyan/spark-warehouse/pracownicy

/home/jovyan/spark-warehouse/pracownicy:
_delta_log  part-00000-96fdb431-5888-4c92-b901-ce1686a76c9d-c000.snappy.parquet

/home/jovyan/spark-warehouse/pracownicy/_delta_log:
00000000000000000000.json
00000000000000000001.json
00000000000000000002.json
00000000000000000003.json
00000000000000000004.json
00000000000000000005.json
00000000000000000006.json
00000000000000000007.json
00000000000000000008.json
00000000000000000009.json
00000000000000000010.checkpoint.parquet
00000000000000000010.json
00000000000000000011.checkpoint.parquet
00000000000000000011.json
00000000000000000012.json
00000000000000000013.json
00000000000000000014.json
00000000000000000015.json
00000000000000000016.json
00000000000000000017.json
00000000000000000018.json
00000000000000000019.json
00000000000000000020.checkpoint.parquet
00000000000000000020.json
00000000000000000021.json
00000000000000000022.json
00000000000000000023.json
00000000000000000024.json
00000000000000000025.json
00000000000000000026.json
0000

In [110]:
spark.sql('SELECT * FROM pracownicy VERSION AS OF 5').show()

Py4JJavaError: An error occurred while calling o654.showString.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 1195.0 failed 1 times, most recent failure: Lost task 0.0 in stage 1195.0 (TID 51210) (78df51bbe6ba executor driver): org.apache.spark.SparkFileNotFoundException: File file:/home/jovyan/spark-warehouse/pracownicy/part-00002-79b30019-a3e4-41f8-a223-c0f90ea1b29d-c000.snappy.parquet does not exist
It is possible the underlying files have been updated. You can explicitly invalidate the cache in Spark by running 'REFRESH TABLE tableName' command in SQL or by recreating the Dataset/DataFrame involved.
	at org.apache.spark.sql.errors.QueryExecutionErrors$.readCurrentFileNotFoundError(QueryExecutionErrors.scala:780)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.org$apache$spark$sql$execution$datasources$FileScanRDD$$anon$$readCurrentFile(FileScanRDD.scala:220)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.